# 日本語指示分類モデルの学習・評価

GPU・APIキー不要。セルを上から実行してください。文字TF-IDFのクラス重心を学習します。
月面基地用データの評価は20例中19例。汎用的な日本語理解の精度を保証するものではありません。

In [ ]:
from pathlib import Path
import json, os, sys
root = Path("/content/pai_intent")
root.mkdir(exist_ok=True)
source_files = {'app/intent.py': '"""Character TF-IDF nearest-centroid classifier; no external dependencies."""\nimport collections\nimport math\nimport unicodedata\n\n\ndef features(text):\n    text = \'\'.join(unicodedata.normalize(\'NFKC\', text).lower().split())\n    return collections.Counter(text[i:i+n] for n in (1, 2, 3)\n                               for i in range(len(text)-n+1))\n\n\ndef normalize(vector):\n    norm = math.sqrt(sum(v*v for v in vector.values())) or 1\n    return {k: v/norm for k, v in vector.items()}\n\n\ndef fit(rows):\n    docs = [features(row[\'text\']) for row in rows]\n    df = collections.Counter(k for doc in docs for k in doc)\n    idf = {k: math.log((len(docs)+1)/(v+1))+1 for k, v in df.items()}\n    sums = {}\n    for row, doc in zip(rows, docs):\n        vec = normalize({k: (1+math.log(v))*idf[k] for k, v in doc.items()})\n        sums.setdefault(row[\'label\'], collections.Counter()).update(vec)\n    return {\'version\': 1, \'idf\': idf,\n            \'centroids\': {label: normalize(vec) for label, vec in sums.items()},\n            \'threshold\': 0.30, \'margin\': 0.08}\n\n\ndef predict(model, text):\n    counts = features(text)\n    vec = normalize({k: (1+math.log(v))*model[\'idf\'][k]\n                     for k, v in counts.items() if k in model[\'idf\']})\n    scores = sorted(((sum(v*centroid.get(k, 0) for k, v in vec.items()), label)\n                     for label, centroid in model[\'centroids\'].items()), reverse=True)\n    score, label = scores[0]\n    margin = score-scores[1][0]\n    # This prototype does not understand negation, multiple destinations, or order.\n    # Reject these explicitly rather than interpreting them as positive commands.\n    unsupported = any(term in text for term in (\'ない\', \'ません\', \'なく\', \'以外\', \'禁止\', \'から\', \'経由\', \'寄って\', \'その後\', \'それから\'))\n    groups = [(\'管制室\', \'コントロールルーム\', \'司令室\'),\n              (\'資材庫\', \'物資庫\', \'ストックヤード\'),\n              (\'研究ラボ\', \'実験室\', \'研究区画\'),\n              (\'充電ドック\', \'帰還地点\', \'ホーム\', \'基地に戻\')]\n    multi = sum(any(word in text for word in group) for group in groups) > 1\n    accepted = score >= model[\'threshold\'] and margin >= model[\'margin\'] and not unsupported and not multi\n    return {\'label\': label if accepted else \'unknown\', \'candidate\': label,\n            \'similarity\': round(score, 4), \'margin\': round(margin, 4)}\n', 'app/train.py': "import json\nfrom pathlib import Path\nfrom intent import fit, predict\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef train(root=ROOT):\n    rows = json.loads((root/'data/commands.json').read_text())\n    model = fit(rows['train'])\n    (root/'models').mkdir(exist_ok=True)\n    (root/'models/intent.json').write_text(json.dumps(model, ensure_ascii=False))\n    results = [dict(row, prediction=predict(model, row['text'])) for row in rows['test']]\n    correct = sum(row['label'] == row['prediction']['label'] for row in results)\n    report = {'correct': correct, 'total': len(results), 'examples': results,\n              'note': 'Small manually authored holdout set; not evidence of general Japanese understanding.'}\n    (root/'results').mkdir(exist_ok=True)\n    (root/'results/intent_evaluation.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))\n    print(f'Intent evaluation: {correct}/{len(results)} (including rejection cases)')\n    return report\n\n\nif __name__ == '__main__':\n    train()\n", 'data/commands.json': '{\n  "train": [\n    {\n      "text": "管制室に行って",\n      "label": "reception"\n    },\n    {\n      "text": "管制室へ向かってください",\n      "label": "reception"\n    },\n    {\n      "text": "管制室まで物資を届けて",\n      "label": "reception"\n    },\n    {\n      "text": "管制室へ配送をお願いします",\n      "label": "reception"\n    },\n    {\n      "text": "目的地は管制室です",\n      "label": "reception"\n    },\n    {\n      "text": "管制室に移動して",\n      "label": "reception"\n    },\n    {\n      "text": "コントロールルームに行って",\n      "label": "reception"\n    },\n    {\n      "text": "コントロールルームへ向かってください",\n      "label": "reception"\n    },\n    {\n      "text": "コントロールルームまで物資を届けて",\n      "label": "reception"\n    },\n    {\n      "text": "コントロールルームへ配送をお願いします",\n      "label": "reception"\n    },\n    {\n      "text": "目的地はコントロールルームです",\n      "label": "reception"\n    },\n    {\n      "text": "コントロールルームに移動して",\n      "label": "reception"\n    },\n    {\n      "text": "司令室に行って",\n      "label": "reception"\n    },\n    {\n      "text": "司令室へ向かってください",\n      "label": "reception"\n    },\n    {\n      "text": "司令室まで物資を届けて",\n      "label": "reception"\n    },\n    {\n      "text": "司令室へ配送をお願いします",\n      "label": "reception"\n    },\n    {\n      "text": "目的地は司令室です",\n      "label": "reception"\n    },\n    {\n      "text": "司令室に移動して",\n      "label": "reception"\n    },\n    {\n      "text": "資材庫に行って",\n      "label": "warehouse"\n    },\n    {\n      "text": "資材庫へ向かってください",\n      "label": "warehouse"\n    },\n    {\n      "text": "資材庫まで物資を届けて",\n      "label": "warehouse"\n    },\n    {\n      "text": "資材庫へ配送をお願いします",\n      "label": "warehouse"\n    },\n    {\n      "text": "目的地は資材庫です",\n      "label": "warehouse"\n    },\n    {\n      "text": "資材庫に移動して",\n      "label": "warehouse"\n    },\n    {\n      "text": "物資庫に行って",\n      "label": "warehouse"\n    },\n    {\n      "text": "物資庫へ向かってください",\n      "label": "warehouse"\n    },\n    {\n      "text": "物資庫まで物資を届けて",\n      "label": "warehouse"\n    },\n    {\n      "text": "物資庫へ配送をお願いします",\n      "label": "warehouse"\n    },\n    {\n      "text": "目的地は物資庫です",\n      "label": "warehouse"\n    },\n    {\n      "text": "物資庫に移動して",\n      "label": "warehouse"\n    },\n    {\n      "text": "ストックヤードに行って",\n      "label": "warehouse"\n    },\n    {\n      "text": "ストックヤードへ向かってください",\n      "label": "warehouse"\n    },\n    {\n      "text": "ストックヤードまで物資を届けて",\n      "label": "warehouse"\n    },\n    {\n      "text": "ストックヤードへ配送をお願いします",\n      "label": "warehouse"\n    },\n    {\n      "text": "目的地はストックヤードです",\n      "label": "warehouse"\n    },\n    {\n      "text": "ストックヤードに移動して",\n      "label": "warehouse"\n    },\n    {\n      "text": "研究ラボに行って",\n      "label": "workshop"\n    },\n    {\n      "text": "研究ラボへ向かってください",\n      "label": "workshop"\n    },\n    {\n      "text": "研究ラボまで物資を届けて",\n      "label": "workshop"\n    },\n    {\n      "text": "研究ラボへ配送をお願いします",\n      "label": "workshop"\n    },\n    {\n      "text": "目的地は研究ラボです",\n      "label": "workshop"\n    },\n    {\n      "text": "研究ラボに移動して",\n      "label": "workshop"\n    },\n    {\n      "text": "実験室に行って",\n      "label": "workshop"\n    },\n    {\n      "text": "実験室へ向かってください",\n      "label": "workshop"\n    },\n    {\n      "text": "実験室まで物資を届けて",\n      "label": "workshop"\n    },\n    {\n      "text": "実験室へ配送をお願いします",\n      "label": "workshop"\n    },\n    {\n      "text": "目的地は実験室です",\n      "label": "workshop"\n    },\n    {\n      "text": "実験室に移動して",\n      "label": "workshop"\n    },\n    {\n      "text": "研究区画に行って",\n      "label": "workshop"\n    },\n    {\n      "text": "研究区画へ向かってください",\n      "label": "workshop"\n    },\n    {\n      "text": "研究区画まで物資を届けて",\n      "label": "workshop"\n    },\n    {\n      "text": "研究区画へ配送をお願いします",\n      "label": "workshop"\n    },\n    {\n      "text": "目的地は研究区画です",\n      "label": "workshop"\n    },\n    {\n      "text": "研究区画に移動して",\n      "label": "workshop"\n    },\n    {\n      "text": "充電ドックに行って",\n      "label": "home"\n    },\n    {\n      "text": "充電ドックへ向かってください",\n      "label": "home"\n    },\n    {\n      "text": "充電ドックまで物資を届けて",\n      "label": "home"\n    },\n    {\n      "text": "充電ドックへ配送をお願いします",\n      "label": "home"\n    },\n    {\n      "text": "目的地は充電ドックです",\n      "label": "home"\n    },\n    {\n      "text": "充電ドックに移動して",\n      "label": "home"\n    },\n    {\n      "text": "帰還地点に行って",\n      "label": "home"\n    },\n    {\n      "text": "帰還地点へ向かってください",\n      "label": "home"\n    },\n    {\n      "text": "帰還地点まで物資を届けて",\n      "label": "home"\n    },\n    {\n      "text": "帰還地点へ配送をお願いします",\n      "label": "home"\n    },\n    {\n      "text": "目的地は帰還地点です",\n      "label": "home"\n    },\n    {\n      "text": "帰還地点に移動して",\n      "label": "home"\n    },\n    {\n      "text": "ホームに行って",\n      "label": "home"\n    },\n    {\n      "text": "ホームへ向かってください",\n      "label": "home"\n    },\n    {\n      "text": "ホームまで物資を届けて",\n      "label": "home"\n    },\n    {\n      "text": "ホームへ配送をお願いします",\n      "label": "home"\n    },\n    {\n      "text": "目的地はホームです",\n      "label": "home"\n    },\n    {\n      "text": "ホームに移動して",\n      "label": "home"\n    },\n    {\n      "text": "充電ドックに戻って",\n      "label": "home"\n    },\n    {\n      "text": "基地に戻って",\n      "label": "home"\n    },\n    {\n      "text": "ホームに帰って",\n      "label": "home"\n    },\n    {\n      "text": "ドックへ帰還して",\n      "label": "home"\n    }\n  ],\n  "test": [\n    {\n      "text": "管制室まで届けてほしい",\n      "label": "reception"\n    },\n    {\n      "text": "司令室に向かって",\n      "label": "reception"\n    },\n    {\n      "text": "コントロールルームへ移動をお願い",\n      "label": "reception"\n    },\n    {\n      "text": "資材庫まで運んで",\n      "label": "warehouse"\n    },\n    {\n      "text": "物資庫に配達してほしい",\n      "label": "warehouse"\n    },\n    {\n      "text": "ストックヤードへ移動しよう",\n      "label": "warehouse"\n    },\n    {\n      "text": "研究ラボまで届けよう",\n      "label": "workshop"\n    },\n    {\n      "text": "実験室へ配達して",\n      "label": "workshop"\n    },\n    {\n      "text": "研究区画に向かおう",\n      "label": "workshop"\n    },\n    {\n      "text": "充電ドックへ戻ろう",\n      "label": "home"\n    },\n    {\n      "text": "ホームまで帰還して",\n      "label": "home"\n    },\n    {\n      "text": "基地に戻ってください",\n      "label": "home"\n    },\n    {\n      "text": "こんにちは",\n      "label": "unknown"\n    },\n    {\n      "text": "天気を教えて",\n      "label": "unknown"\n    },\n    {\n      "text": "食堂へ行って",\n      "label": "unknown"\n    },\n    {\n      "text": "管制室には行かないで",\n      "label": "unknown"\n    },\n    {\n      "text": "資材庫と研究ラボに行って",\n      "label": "unknown"\n    },\n    {\n      "text": "研究ラボ以外へ行って",\n      "label": "unknown"\n    },\n    {\n      "text": "充電ドックから管制室へ移動して",\n      "label": "unknown"\n    },\n    {\n      "text": "司令室とホームに行って",\n      "label": "unknown"\n    }\n  ]\n}'}
for name, content in source_files.items():
    target = root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
(root / "models").mkdir(exist_ok=True)
(root / "results").mkdir(exist_ok=True)
sys.path.insert(0, str(root / "app"))


## 学習データを編集する

次のセルの前に `data/commands.json` の学習例を編集しても構いません。新しい表現を追加した場合は、学習に入れない別の評価文も作り、過学習を確認してください。目的地クラス名はMac側の制御コードと一致させます。

In [ ]:
from train import train
report = train(root)
for row in report["examples"]:
    print(row["text"], "期待:", row["label"], "判定:", row["prediction"])


In [ ]:
from intent import predict
model = json.loads((root / "models/intent.json").read_text())
text = "管制室まで物資を届けて" # この文字列を変えて試す
predict(model, text)


## 学習結果をダウンロード

ZIPにはデータ・モデル・評価結果が入ります。Mac側は起動時に再学習するので、特に `data/commands.json` をプロジェクトの同名ファイルに反映してください。

In [ ]:
import zipfile
from google.colab import files
bundle = "/content/trained_intent_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["data/commands.json", "models/intent.json", "results/intent_evaluation.json"]:
        z.write(root / name, name)
files.download(bundle)
